# 03 损失函数与梯度下降

上一节我们知道了：多层感知机通过隐藏层，可以表达比单个感知机更复杂的规则。

但还有一个更关键的问题没有解决：**模型里的参数到底怎么变好？**

这一节就讲清楚这条逻辑：

```text
预测结果
-> 和真实答案比较
-> 得到错误大小
-> 用损失函数表示错误
-> 用梯度判断参数该往哪里改
-> 用梯度下降更新参数
```

## 1. 为什么需要损失函数

神经网络训练时，模型会先给出一个预测值：

$$
\hat{y}=f_{\theta}(x)
$$

其中 $x$ 是输入，$\hat{y}$ 是预测结果，$\theta$ 表示模型中所有参数。

但训练数据里还有真实答案：

$$
y
$$

学习的第一步，就是比较：

$$
\hat{y} \quad \text{和} \quad y
$$

如果预测值和真实值差得很远，说明模型现在不好；如果差得很近，说明模型比较好。

但是“差得远”和“差得近”只是口头描述，计算机没法直接优化一句话。我们必须把错误变成一个明确的数字，这个数字就叫损失。

## 2. 损失函数是什么

损失函数就是用来衡量模型预测有多错的函数。

可以写成：

$$
\mathcal{L}(\hat{y},y)
$$

其中：

- $\hat{y}$ 是模型预测。
- $y$ 是真实答案。
- $\mathcal{L}$ 是损失函数。

损失函数的输出是一个数字：

$$
\mathcal{L} \in \mathbb{R}
$$

这个数字越小，说明模型越好；这个数字越大，说明模型越差。

所以训练模型的目标可以写成：

$$
\min_{\theta}\mathcal{L}
$$

这句话的意思是：调整参数 $\theta$，让损失函数尽可能小。

## 3. 用一个最简单的例子理解损失

假设我们在做房价预测，真实价格是：

$$
y=100
$$

模型预测为：

$$
\hat{y}=90
$$

最直接的错误是：

$$
\hat{y}-y=90-100=-10
$$

但这个错误有正有负。如果另一个模型预测为 $110$，错误是：

$$
110-100=10
$$

一个是 $-10$，一个是 $10$，如果直接相加，可能互相抵消。为了避免这个问题，常用平方误差：

$$
(\hat{y}-y)^2
$$

这样无论预测偏大还是偏小，损失都是正数。

## 4. 均方误差 MSE

回归任务中常见的损失函数是均方误差，英文是 Mean Squared Error，简称 MSE。

如果只有一个样本：

$$
\mathcal{L}=(\hat{y}-y)^2
$$

如果有 $m$ 个样本，就把每个样本的平方误差求平均：

$$
\mathcal{L}=\frac{1}{m}\sum_{i=1}^{m}(\hat{y}^{(i)}-y^{(i)})^2
$$

这里的平均很重要，因为不同批次的样本数量可能不同。求平均之后，损失大小更容易比较。

MSE 的直觉是：预测离真实值越远，惩罚越大。因为误差会被平方，较大的错误会被放大。

## 5. 分类任务的损失直觉

分类任务里，模型通常输出概率。

例如二分类中，模型输出：

$$
\hat{y}=P(y=1\mid x)
$$

如果真实标签是 $1$，我们希望 $\hat{y}$ 越接近 $1$ 越好。

如果真实标签是 $0$，我们希望 $\hat{y}$ 越接近 $0$ 越好。

二分类常用的损失函数是二元交叉熵：

$$
\mathcal{L}=-\left[y\log(\hat{y})+(1-y)\log(1-\hat{y})\right]
$$

这个公式先不用死记。先理解它在惩罚什么：

- 真实是 $1$，但模型给 $\hat{y}$ 很小，损失会很大。
- 真实是 $0$，但模型给 $\hat{y}$ 很大，损失会很大。
- 模型对真实类别越有信心，损失越小。

## 6. 有了损失，为什么还需要梯度

损失函数告诉我们：模型现在错得有多严重。

但它还没有告诉我们：参数应该怎么改。

比如某个参数是 $w$，当前损失是：

$$
\mathcal{L}(w)
$$

我们需要知道：如果把 $w$ 稍微调大一点，损失会变大还是变小？

这个信息由导数给出：

$$
\frac{d\mathcal{L}}{dw}
$$

导数可以理解成：参数 $w$ 变化一点点时，损失函数会怎么变化。

## 7. 梯度是什么

如果只有一个参数 $w$，我们说导数：

$$
\frac{d\mathcal{L}}{dw}
$$

如果模型有很多参数，比如：

$$
\theta=[w_1,w_2,b]
$$

那就要分别看损失对每个参数的导数：

$$
\frac{\partial \mathcal{L}}{\partial w_1}, \quad
\frac{\partial \mathcal{L}}{\partial w_2}, \quad
\frac{\partial \mathcal{L}}{\partial b}
$$

把这些偏导数组成一个向量，就叫梯度：

$$
\nabla_{\theta}\mathcal{L}=\left[
\frac{\partial \mathcal{L}}{\partial w_1},
\frac{\partial \mathcal{L}}{\partial w_2},
\frac{\partial \mathcal{L}}{\partial b}
\right]
$$

梯度的方向，是损失函数上升最快的方向。

既然训练的目标是让损失变小，就应该往梯度的反方向走。

## 8. 梯度下降是什么

梯度下降就是沿着梯度的反方向更新参数。

参数更新公式是：

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}
$$

其中：

- $\theta$ 是模型参数。
- $\mathcal{L}$ 是损失函数。
- $\nabla_{\theta}\mathcal{L}$ 是损失对参数的梯度。
- $\eta$ 是学习率。

这个公式的意思很朴素：

1. 先看当前参数让模型错得有多严重。
2. 再看每个参数往哪个方向改会让错误变大。
3. 然后反着来，让错误变小。

## 9. 用一个一维例子理解梯度下降

假设损失函数是：

$$
\mathcal{L}(w)=(w-3)^2
$$

这个函数在 $w=3$ 时最小，因为：

$$
\mathcal{L}(3)=(3-3)^2=0
$$

它的导数是：

$$
\frac{d\mathcal{L}}{dw}=2(w-3)
$$

如果当前 $w=0$，导数是：

$$
\frac{d\mathcal{L}}{dw}=2(0-3)=-6
$$

梯度是负数，说明如果 $w$ 增大，损失会下降。所以更新时：

$$
w \leftarrow w-\eta(-6)
$$

也就是：

$$
w \leftarrow w+6\eta
$$

参数 $w$ 会向 $3$ 靠近。

## 10. 学习率是什么

学习率 $\eta$ 决定每次更新参数时迈多大的步子。

如果学习率太小，更新很慢：

$$
\theta \leftarrow \theta-0.0001\nabla_{\theta}\mathcal{L}
$$

模型可能要训练很久才有明显变化。

如果学习率太大，步子迈得太猛，可能直接跨过最低点，甚至越跑越远：

$$
\theta \leftarrow \theta-10\nabla_{\theta}\mathcal{L}
$$

所以学习率不是越大越好，也不是越小越好。它控制的是训练的步幅。

## 11. 一个参数更新到底发生了什么

神经网络训练中的一次更新，可以拆成四步：

1. 前向传播：用当前参数算预测值。

$$
\hat{y}=f_{\theta}(x)
$$

2. 计算损失：比较预测值和真实值。

$$
\mathcal{L}=\mathcal{L}(\hat{y},y)
$$

3. 计算梯度：判断每个参数对损失的影响。

$$
\nabla_{\theta}\mathcal{L}
$$

4. 更新参数：沿着让损失下降的方向走一步。

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}
$$

这就是训练循环的核心。后面写 PyTorch 时，代码只是这些概念的实现。

## 12. 为什么这还不是反向传播

这一节讲的是：有了损失函数之后，参数应该沿着梯度反方向更新。

但还有一个问题没有解决：神经网络可能有很多层，参数也很多，梯度到底怎么高效算出来？

比如两层网络：

$$
\mathbf{h}=\phi(\mathbf{W}_1\mathbf{x}+\mathbf{b}_1)
$$

$$
\hat{y}=g(\mathbf{W}_2\mathbf{h}+\mathbf{b}_2)
$$

损失是：

$$
\mathcal{L}=\mathcal{L}(\hat{y},y)
$$

我们需要计算：

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1}, \quad
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_1}, \quad
\frac{\partial \mathcal{L}}{\partial \mathbf{W}_2}, \quad
\frac{\partial \mathcal{L}}{\partial \mathbf{b}_2}
$$

这些梯度不是凭空来的，而是通过链式法则一层层算出来的。这个过程就叫反向传播。

所以本节先讲清楚“为什么要梯度下降”，下一节再讲“梯度怎么通过反向传播算出来”。

## 13. 本节总结

这一节的逻辑链是：

```text
模型会预测
-> 预测会出错
-> 损失函数把错误变成一个数字
-> 训练目标是让损失变小
-> 梯度告诉我们参数往哪里变会让损失增大
-> 所以沿着梯度反方向更新参数
-> 这就是梯度下降
```

先记住三个公式：

损失函数：

$$
\mathcal{L}(\hat{y},y)
$$

梯度：

$$
\nabla_{\theta}\mathcal{L}
$$

梯度下降：

$$
\theta \leftarrow \theta-\eta\nabla_{\theta}\mathcal{L}
$$

下一节进入反向传播，重点讲清楚：为什么链式法则能让神经网络知道每个参数该怎么改。